# 04 — Poisson Baseline and Overdispersion

The study's model is Negative Binomial. This notebook fits the Poisson model first, on purpose.

Poisson regression is the standard starting point for count data, but it assumes the conditional
variance equals the conditional mean. The Negative Binomial model relaxes that assumption by adding
a dispersion parameter. Fitting Poisson first and measuring how badly the assumption fails turns
the choice of Negative Binomial into a documented finding from this dataset rather than an
assertion.

Model form:

```
Dengue Cases ~ Poisson(mu)
log(mu) = intercept + b1 * density + year terms + log(Population)
```

Model logic lives in `src/models.py` and is covered by `tests/test_models.py`.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.indicators import build_indicator_panel
from src.models import (
    build_design_matrix,
    fit_poisson_baseline,
    fit_quasi_poisson,
    overdispersion_index,
    deviance_ratio,
    overdispersion_report,
    coefficient_table,
)

ROOT = Path.cwd().parent
VALIDATED = ROOT / "data" / "04_validated"
ANALYTICAL = ROOT / "outputs" / "analytical"
ANALYTICAL.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

panel = pd.read_csv(VALIDATED / "lgu_year_panel.csv")
indicators = build_indicator_panel(panel)
print(f"Panel: {len(panel)} LGU-year observations")

Panel: 85 LGU-year observations


## 1. Specification

**Population enters as an offset, not as a predictor.** Its coefficient is fixed at one, which
turns a model of counts into a model of rates. Without it, the model would learn that large LGUs
have more cases, which is true and useless. With it, the question becomes whether an LGU has more
cases *than its population would lead you to expect*.

**Density is expressed per 1,000 persons per square kilometre.** The raw values run from about
6,300 to about 76,700, so a coefficient on the raw scale would be reported in the fourth or fifth
decimal place. Dividing by 1,000 makes it readable as the effect of a thousand extra persons per
square kilometre.

**Year enters as dummy variables, with 2021 as the reference.** NCR case totals over the five years
are 10,493 / 43,753 / 23,678 / 37,225 / 45,014 — up, down, then up again. A single linear year term
would force a straight line through a shape that is not straight. A dummy per year lets each year
find its own level, which is what year is doing in this model: absorbing region-wide epidemic
variation so that the density coefficient is estimated within years rather than across them.

The year terms are not interpreted as a trend and are not projected forward.

In [2]:
design = build_design_matrix(indicators, year_as="factor")
print("Design matrix columns:", list(design.columns))
design.head()

Design matrix columns: ['const', 'Density per 1000', 'Year_2022', 'Year_2023', 'Year_2024', 'Year_2025']


,const,Density per 1000,Year_2022,Year_2023,Year_2024,Year_2025
0,1.0,30.007603,0.0,0.0,0.0,0.0
1,1.0,30.237715,1.0,0.0,0.0,0.0
2,1.0,30.467827,0.0,1.0,0.0,0.0
3,1.0,30.697939,0.0,0.0,1.0,0.0
4,1.0,30.928051,0.0,0.0,0.0,1.0


## 2. Fit the Poisson baseline

In [3]:
poisson_fit = fit_poisson_baseline(indicators, year_as="factor")
print(poisson_fit.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:                            GLM   Df Residuals:                       79
Model Family:                 Poisson   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -7718.6
Date:                Wed, 02 Sep 2026   Deviance:                       14683.
Time:                        22:03:48   Pearson chi2:                 1.57e+04
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -7.1200      0.010  

## 3. Reading the coefficients

A Poisson or Negative Binomial coefficient is read after exponentiating. `exp(b)` is the
multiplicative effect on the expected case rate for a one-unit increase in that predictor, holding
the others fixed — an incidence rate ratio. A ratio of 1.05 means five percent higher expected
incidence; 0.95 means five percent lower.

The p-values printed above should **not** be reported yet. Section 5 shows why.

In [4]:
coefficients = coefficient_table(poisson_fit)
coefficients.round(4)

,Coefficient,Std. Error,z,p-value,Rate Ratio,CI Lower,CI Upper
Term,,,,,,,
const,-7.1200,0.0104,-682.3745,0.0,0.0008,0.0008,0.0008
Density per 1000,-0.0017,0.0001,-12.9392,0.0,0.9983,0.9981,0.9986
Year_2022,1.4188,0.0109,130.5269,0.0,4.1323,4.0452,4.2213
Year_2023,0.7960,0.0117,67.8751,0.0,2.2167,2.1663,2.2683
Year_2024,1.2399,0.0111,112.1693,0.0,3.4552,3.3811,3.5309
Year_2025,1.4215,0.0108,131.1071,0.0,4.1433,4.0562,4.2323


## 4. Overdispersion index

The Poisson assumption is that the conditional variance equals the conditional mean. The
overdispersion index tests it directly:

```
overdispersion index = Pearson chi-square / residual degrees of freedom
```

Under a correctly specified Poisson model the index is close to 1. Above 1 means the data varies
more than Poisson allows. The residual deviance divided by the same degrees of freedom is reported
alongside it as a second reading, so the conclusion does not rest on one statistic.

Dengue counts are expected to be overdispersed. Cases cluster in space and time, one outbreak
seeds the next, and reporting intensity varies — all of which produce years far above and far below
the LGU's own typical level. The index puts a number on how much.

In [5]:
report = overdispersion_report(poisson_fit)

for key, value in report.items():
    print(f"{key:>22}: {value:,.4f}" if isinstance(value, float) else f"{key:>22}: {value}")

          observations: 85
            parameters: 6
              df_resid: 79
          pearson_chi2: 15,716.9088
  overdispersion_index: 198.9482
              deviance: 14,683.0810
        deviance_ratio: 185.8618
        log_likelihood: -7,718.5907
                   aic: 15,449.1814
               verdict: Substantial overdispersion; Negative Binomial is warranted


In [6]:
# The same story in raw terms: for a Poisson variable the variance and the mean
# would be roughly equal.
cases = indicators["Dengue Cases"]
print(f"Mean case count:      {cases.mean():,.1f}")
print(f"Variance:             {cases.var():,.1f}")
print(f"Variance / mean:      {cases.var() / cases.mean():,.1f}")

Mean case count:      1,884.3
Variance:             4,524,064.3
Variance / mean:      2,401.0


## 5. What overdispersion does to the standard errors

Overdispersion does not bias the coefficients — the Poisson estimates stay consistent. It attacks
the standard errors, which are computed under a variance assumption the data does not satisfy.
They come out too small, so every predictor looks more precisely estimated than it is, and p-values
that look decisive are not.

Rescaling the standard errors by the Pearson dispersion shows the size of that gap. This is a
diagnostic, not the study's model: the Negative Binomial fit in notebook 05 handles the extra
variance properly rather than patching the standard errors after the fact.

In [7]:
quasi_fit = fit_quasi_poisson(indicators, year_as="factor")

se_comparison = pd.DataFrame({
    "Coefficient": poisson_fit.params,
    "SE (Poisson)": poisson_fit.bse,
    "SE (dispersion-scaled)": quasi_fit.bse,
    "Inflation factor": quasi_fit.bse / poisson_fit.bse,
    "p (Poisson)": poisson_fit.pvalues,
    "p (dispersion-scaled)": quasi_fit.pvalues,
})
se_comparison.round(4)

,Coefficient,SE (Poisson),SE (dispersion-scaled),Inflation factor,p (Poisson),p (dispersion-scaled)
const,-7.1200,0.0104,0.1472,14.1049,0.0,0.000
Density per 1000,-0.0017,0.0001,0.0018,14.1049,0.0,0.359
Year_2022,1.4188,0.0109,0.1533,14.1049,0.0,0.000
Year_2023,0.7960,0.0117,0.1654,14.1049,0.0,0.000
Year_2024,1.2399,0.0111,0.1559,14.1049,0.0,0.000
Year_2025,1.4215,0.0108,0.1529,14.1049,0.0,0.000


## 6. Robustness check: year as a linear term

The same model with a single linear year term instead of year dummies, for comparison only. If the
linear version fits far worse, that supports treating year as a set of levels rather than a trend.

In [8]:
linear_year_fit = fit_poisson_baseline(indicators, year_as="numeric")

pd.DataFrame({
    "Year as dummies": {
        "overdispersion index": overdispersion_index(poisson_fit),
        "deviance ratio": deviance_ratio(poisson_fit),
        "AIC": poisson_fit.aic,
        "log-likelihood": poisson_fit.llf,
    },
    "Year as linear term": {
        "overdispersion index": overdispersion_index(linear_year_fit),
        "deviance ratio": deviance_ratio(linear_year_fit),
        "AIC": linear_year_fit.aic,
        "log-likelihood": linear_year_fit.llf,
    },
}).round(2)

,Year as dummies,Year as linear term
overdispersion index,198.95,476.80
deviance ratio,185.86,411.47
AIC,15449.18,34500.79
log-likelihood,-7718.59,-17247.40


## 7. Unit tests: the model layer

`tests/test_models.py` covers the design matrix (intercept, scaled density, four year dummies with
2021 as reference), the offset being exactly `log(Population)`, and the overdispersion index. Two
of those tests matter most:

- On synthetic data drawn from a **genuine Poisson process**, the index must come back near 1. A
  diagnostic that always reports overdispersion would be worthless.
- On synthetic data drawn with **known extra variance**, the index must come back large.

Together they show the index responds to the data rather than to the code.

In [9]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_models.py"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout[-3500:])

============================= test session starts =============================
platform win32 -- Python 3.14.2, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\Mae\OneDrive\Documents\GitHub\Capstone 2\capstone2_dengue_ncr\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Mae\OneDrive\Documents\GitHub\Capstone 2\capstone2_dengue_ncr
configfile: pytest.ini
plugins: anyio-4.14.2
collecting ... collected 24 items

tests/test_models.py::test_design_matrix_has_intercept_density_and_four_year_dummies PASSED [  4%]
tests/test_models.py::test_density_is_scaled_to_thousands PASSED         [  8%]
tests/test_models.py::test_numeric_year_option_gives_a_single_year_term PASSED [ 12%]
tests/test_models.py::test_unknown_year_option_is_rejected PASSED        [ 16%]
tests/test_models.py::test_offset_is_log_population PASSED               [ 20%]
tests/test_models.py::test_offset_makes_the_model_estimate_rates_not_counts PASSED [ 25%]
tests/test_models.py::test_model_uses_all_85_observations PASS

## 8. Write the analytical outputs

In [10]:
coefficients.to_csv(ANALYTICAL / "poisson_baseline_coefficients.csv")

with open(ANALYTICAL / "poisson_baseline_summary.txt", "w") as f:
    f.write(str(poisson_fit.summary()))

with open(ANALYTICAL / "overdispersion.json", "w") as f:
    json.dump(report, f, indent=2)

print("Written:")
for name in ["poisson_baseline_coefficients.csv", "poisson_baseline_summary.txt", "overdispersion.json"]:
    print(f"  outputs/analytical/{name}")

Written:
  outputs/analytical/poisson_baseline_coefficients.csv
  outputs/analytical/poisson_baseline_summary.txt
  outputs/analytical/overdispersion.json


## Summary

- Poisson baseline fitted on all 85 LGU-year observations, with `log(Population)` as the offset so
  the model estimates rates rather than counts.
- The overdispersion index is far above 1, so the Poisson variance assumption does not hold for this
  data. Written up in `docs/overdispersion.md`.
- Standard errors under Poisson are correspondingly too small, which is why the baseline p-values
  are not reported as findings.
- This is the documented basis for the study's Negative Binomial specification.

**Next:** notebook 05 fits the Negative Binomial model, tests the dispersion parameter against the
Poisson fit, and produces the structural risk ranking that feeds the three-tier priority rule.